# KAE GPU Agent (Colab)

Поднимает **ollama** с vision/coder-моделью на бесплатном Colab-GPU и пробрасывает
порт наружу через cloudflared. Получившийся URL добавляется в KAE как обычный
агент в менеджере агентов (RFC 0021 §3.1) — GPU-скорость для `tikz_vectorization`
без своего железа и без ухода данных в проприетарный API.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.
Затем Runtime → Run all. URL агента появится в последней ячейке.

In [ ]:
# 1. Проверка GPU и установка ollama
!nvidia-smi -L || echo 'GPU не выбран: Runtime -> Change runtime type -> T4 GPU'
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Запуск ollama на всех интерфейсах
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'])
time.sleep(6)
!curl -s http://127.0.0.1:11434/api/tags && echo '\nollama up'

In [ ]:
# 3. Vision-модель для схема→TikZ. Выбери по своему GPU:
#    T4 (бесплатно, 16GB):  llama3.2-vision:11b  или  minicpm-v
#    A100 (Colab Pro, 40GB): qwen2.5vl:32b  (лучшее качество)
MODEL = "llama3.2-vision:11b"   # поменяй при A100 на qwen2.5vl:32b
!ollama pull {MODEL}
print("Vision-модель готова:", MODEL)

In [ ]:
# 4. Туннель наружу (cloudflared) — публичный HTTPS-URL к ollama
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, itertools
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in itertools.islice(proc.stdout, 200):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('\n' + '=' * 60)
print('URL АГЕНТА:', url)
print('=' * 60)
print('Добавь в KAE (менеджер агентов): host =', url)
print('Vision-модель:', MODEL, '— картинка схемы -> TikZ')
print('Не закрывай эту вкладку — агент живёт, пока работает ноутбук.')

In [ ]:
# 5. Держать сессию живой (Colab выгружает простаивающие runtime)
import time
print('Агент активен. Держи вкладку открытой.')
while True:
    time.sleep(300)